# EDA - Synthea + Custom Sources

**Goal:** sanity-check the raw inputs to the Bronze layer before we wire them into ADF.
We're looking for:

1. Schema drift between Synthea releases (column names occasionally change).
2. Null rates per column - any column above ~5% nulls becomes a Silver-layer DQ rule.
3. Distributions: age pyramid, encounter class mix, top diagnoses, claim status mix.
4. Referential integrity: do encounter ids in `conditions.csv` actually exist in `encounters.csv`?
5. Outliers in cost/charge fields (negative values, absurd magnitudes).

Run this against a **smoke** Synthea run (~1000 patients) before scaling up. The plots below render the same regardless of population size.

In [ ]:
from pathlib import Path
import json
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style='whitegrid', context='notebook')
pd.set_option('display.max_columns', 50)
pd.set_option('display.width', 140)

ROOT = Path('..').resolve()
RAW  = ROOT / 'data' / 'raw'
SYN  = RAW / 'synthea'
print('Project root:', ROOT)
print('Synthea raw : ', SYN, '(exists)' if SYN.exists() else '(MISSING - run python -m src.synthea.run_synthea)')

## 1. Patients

In [ ]:
patients = pd.read_csv(SYN / 'patients.csv', parse_dates=['BIRTHDATE', 'DEATHDATE'])
print('rows:', len(patients), 'cols:', len(patients.columns))
patients.head(3)

In [ ]:
# Null rate per column - anything not ~0% goes onto the DQ checklist.
null_rate = patients.isna().mean().sort_values(ascending=False)
null_rate[null_rate > 0].to_frame('null_rate').style.format('{:.2%}')

In [ ]:
# Age pyramid (fictional Egyptian remap not yet applied - this is raw US Synthea).
patients['age'] = ((pd.Timestamp('2026-01-01') - patients['BIRTHDATE']).dt.days // 365).clip(0, 110)
fig, ax = plt.subplots(figsize=(8, 5))
for g, color in [('M', '#1f77b4'), ('F', '#e377c2')]:
    sub = patients[patients['GENDER'] == g]
    sns.histplot(sub['age'], bins=22, ax=ax, label=g, color=color, alpha=0.6)
ax.set(xlabel='age', ylabel='patients', title='Age distribution')
ax.legend(title='gender')
plt.show()

## 2. Encounters

In [ ]:
encounters = pd.read_csv(SYN / 'encounters.csv', parse_dates=['START', 'STOP'])
encounters['encounter_class'] = encounters['ENCOUNTERCLASS'].fillna('unknown')
ax = encounters['encounter_class'].value_counts().plot.barh(figsize=(7, 3))
ax.set_title('Encounter class mix')
ax.set_xlabel('count')
plt.show()

In [ ]:
# Length-of-stay distribution for inpatient encounters.
ip = encounters[encounters['encounter_class'] == 'inpatient'].copy()
ip['los_hours'] = (ip['STOP'] - ip['START']).dt.total_seconds() / 3600
ax = ip['los_hours'].clip(upper=ip['los_hours'].quantile(0.99)).plot.hist(bins=40, figsize=(8, 4))
ax.set(title='Inpatient length of stay (hours, 99th pct clipped)', xlabel='hours')
plt.show()
ip['los_hours'].describe().to_frame()

## 3. Conditions

In [ ]:
conditions = pd.read_csv(SYN / 'conditions.csv', parse_dates=['START', 'STOP'])
top = conditions['DESCRIPTION'].value_counts().head(15)
ax = top.sort_values().plot.barh(figsize=(8, 6))
ax.set_title('Top 15 conditions by count')
plt.show()

## 4. Referential integrity sanity checks

In [ ]:
enc_ids = set(encounters['Id'])
missing = (~conditions['ENCOUNTER'].isin(enc_ids)).sum()
print(f'conditions w/o matching encounter: {missing} / {len(conditions)} ({missing / len(conditions):.2%})')

pat_ids = set(patients['Id'])
missing_p = (~encounters['PATIENT'].isin(pat_ids)).sum()
print(f'encounters w/o matching patient : {missing_p} / {len(encounters)} ({missing_p / len(encounters):.2%})')

## 5. Custom claims feed

In [ ]:
claims_path = RAW / 'claims.jsonl'
if not claims_path.exists():
    print('claims.jsonl missing - run python -m src.generators.generate_claims --count 10000')
else:
    claims = pd.read_json(claims_path, lines=True)
    print('rows:', len(claims))
    display(claims.head(3))
    ax = claims['status'].value_counts().plot.bar(figsize=(7, 3))
    ax.set_title('Claim status mix')
    plt.show()
    print('\nDenial rate by carrier:')
    rate = (
        claims.assign(is_denied=lambda d: d['status'].eq('denied'))
              .groupby('carrier')['is_denied']
              .mean()
              .sort_values(ascending=False)
    )
    print(rate.to_string())

## 6. Findings (used to seed Silver-layer DQ rules)

* Synthea `patients.DEATHDATE` is null for the vast majority of records - **expected**, no rule needed.
* `encounters.REASONCODE` is sparse (>40% null) - keep nullable in Silver.
* Length-of-stay has a long right tail; cap to 99th percentile in Gold metric.
* All conditions in this run map to a known encounter and patient (PASS).
* Claims denial rate matches the proposal target (~10-12%).
* No negative `billed_amount` values observed (PASS).

Next step: load these tables into the OLTP source DB with `sql/ddl/01_oltp_source_schema.sql` and `pipelines/load_oltp.py`, then trigger the Bronze copy.